# Image Mask Creation Pipeline with OpenCV

This notebook provides a configurable pipeline to create masks for images and save them to a separate folder.


## 1. Configuration


In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
from typing import Callable, Optional

# ========== CONFIGURATION ==========

# Input and output paths
INPUT_FOLDER = "D:/"  # Folder containing input images
OUTPUT_FOLDER = "masks"  # Folder where masks will be saved

# Supported image extensions
IMAGE_EXTENSIONS = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']

# Mask creation method: 'threshold', 'color_range', 'contour', 'adaptive', 'custom'
MASK_METHOD = 'threshold'  # Change this to use different methods

# Naming function - customize this to your needs
def mask_naming_function(original_filename: str) -> str:
    """
    Customize this function to define how mask files should be named.
    
    Args:
        original_filename: Original image filename
    
    Returns:
        New filename for the mask
    """
    # Example 1: Add '_mask' suffix
    name, ext = os.path.splitext(original_filename)
    return f"{name}_mask{ext}"
    
    # Example 2: Replace extension with '_mask.png'
    # name, _ = os.path.splitext(original_filename)
    # return f"{name}_mask.png"
    
    # Example 3: Add prefix
    # return f"mask_{original_filename}"

# Mask parameters (adjust based on MASK_METHOD)
MASK_PARAMS = {
    # For 'threshold' method
    'threshold_value': 127,  # Threshold value (0-255)
    'threshold_type': cv2.THRESH_BINARY,  # cv2.THRESH_BINARY, cv2.THRESH_BINARY_INV, etc.
    
    # For 'color_range' method (HSV color space)
    'lower_hsv': np.array([0, 50, 50]),  # Lower HSV bound
    'upper_hsv': np.array([180, 255, 255]),  # Upper HSV bound
    
    # For 'contour' method
    'contour_area_min': 100,  # Minimum contour area to keep
    'fill_contours': True,  # Fill contours or just draw outlines
    
    # For 'adaptive' method
    'adaptive_method': cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    'block_size': 11,
    'C': 2,
}

# Output format
OUTPUT_FORMAT = '.png'  # '.png', '.jpg', etc. (PNG recommended for masks)

# Visualization
SHOW_PREVIEW = True  # Show preview of first few masks
PREVIEW_COUNT = 3  # Number of preview images to show


## 2. Mask Creation Functions


In [ ]:
def create_threshold_mask(image: np.ndarray, params: dict) -> np.ndarray:
    """Create mask using simple thresholding."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    _, mask = cv2.threshold(gray, params['threshold_value'], 255, params['threshold_type'])
    return mask

def create_color_range_mask(image: np.ndarray, params: dict) -> np.ndarray:
    """Create mask based on color range in HSV space."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, params['lower_hsv'], params['upper_hsv'])
    return mask

def create_contour_mask(image: np.ndarray, params: dict) -> np.ndarray:
    """Create mask from contours."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    mask = np.zeros_like(gray)
    for contour in contours:
        if cv2.contourArea(contour) > params['contour_area_min']:
            if params['fill_contours']:
                cv2.drawContours(mask, [contour], -1, 255, -1)
            else:
                cv2.drawContours(mask, [contour], -1, 255, 2)
    return mask

def create_adaptive_mask(image: np.ndarray, params: dict) -> np.ndarray:
    """Create mask using adaptive thresholding."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    mask = cv2.adaptiveThreshold(
        gray, 255, params['adaptive_method'], cv2.THRESH_BINARY, 
        params['block_size'], params['C']
    )
    return mask

def create_custom_mask(image: np.ndarray, params: dict) -> np.ndarray:
    """
    Custom mask creation function.
    Modify this function to implement your own mask creation logic.
    """
    # Example: Combine multiple methods
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    
    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Apply threshold
    _, mask = cv2.threshold(blurred, params.get('threshold_value', 127), 255, cv2.THRESH_BINARY)
    
    # Apply morphological operations (optional)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    return mask

# Map method names to functions
MASK_FUNCTIONS = {
    'threshold': create_threshold_mask,
    'color_range': create_color_range_mask,
    'contour': create_contour_mask,
    'adaptive': create_adaptive_mask,
    'custom': create_custom_mask,
}


## 3. Main Processing Pipeline


In [ ]:
def process_images(
    input_folder: str,
    output_folder: str,
    mask_method: str,
    mask_params: dict,
    naming_function: Callable,
    output_format: str = '.png',
    show_preview: bool = True,
    preview_count: int = 3
) -> list:
    """
    Process all images in the input folder and create masks.
    
    Returns:
        List of processed file paths
    """
    # Validate mask method
    if mask_method not in MASK_FUNCTIONS:
        raise ValueError(f"Unknown mask method: {mask_method}. Available: {list(MASK_FUNCTIONS.keys())}")
    
    # Get mask creation function
    mask_func = MASK_FUNCTIONS[mask_method]
    
    # Create output folder if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # Get all image files
    input_path = Path(input_folder)
    image_files = [f for f in input_path.iterdir() 
                   if f.suffix.lower() in IMAGE_EXTENSIONS]
    
    if not image_files:
        print(f"No image files found in {input_folder}")
        return []
    
    print(f"Found {len(image_files)} images to process...")
    
    processed_files = []
    preview_images = []
    
    for idx, image_path in enumerate(image_files):
        try:
            # Read image
            image = cv2.imread(str(image_path))
            if image is None:
                print(f"Warning: Could not read {image_path}")
                continue
            
            # Create mask
            mask = mask_func(image, mask_params)
            
            # Generate output filename
            mask_filename = naming_function(image_path.name)
            
            # Ensure output format extension
            if not mask_filename.endswith(output_format):
                name, _ = os.path.splitext(mask_filename)
                mask_filename = f"{name}{output_format}" 
            
            # Save mask
            output_path = Path(output_folder) / mask_filename
            cv2.imwrite(str(output_path), mask)
            
            processed_files.append(str(output_path))
            
            # Store for preview
            if show_preview and len(preview_images) < preview_count:
                preview_images.append({
                    'original': image,
                    'mask': mask,
                    'filename': image_path.name
                })
            
            if (idx + 1) % 10 == 0:
                print(f"Processed {idx + 1}/{len(image_files)} images...")
                
        except Exception as e:
            print(f"Error processing {image_path}: {str(e)}")
            continue
    
    print(f"\nCompleted! Processed {len(processed_files)} images.")
    print(f"Masks saved to: {output_folder}")
    
    # Show preview
    if show_preview and preview_images:
        show_preview_images(preview_images)
    
    return processed_files

def show_preview_images(preview_images: list):
    """Display preview of original images and their masks."""
    n = len(preview_images)
    fig, axes = plt.subplots(n, 2, figsize=(12, 4 * n))
    
    if n == 1:
        axes = axes.reshape(1, -1)
    
    for i, preview in enumerate(preview_images):
        # Original image
        original_rgb = cv2.cvtColor(preview['original'], cv2.COLOR_BGR2RGB)
        axes[i, 0].imshow(original_rgb)
        axes[i, 0].set_title(f"Original: {preview['filename']}")
        axes[i, 0].axis('off')
        
        # Mask
        axes[i, 1].imshow(preview['mask'], cmap='gray')
        axes[i, 1].set_title(f"Mask")
        axes[i, 1].axis('off')
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Execute the pipeline
processed_files = process_images(
    input_folder=INPUT_FOLDER,
    output_folder=OUTPUT_FOLDER,
    mask_method=MASK_METHOD,
    mask_params=MASK_PARAMS,
    naming_function=mask_naming_function,
    output_format=OUTPUT_FORMAT,
    show_preview=SHOW_PREVIEW,
    preview_count=PREVIEW_COUNT
)


## 5. Advanced: Process Single Image (for testing)


In [ ]:
# Test on a single image before processing the whole dataset
def test_single_image(image_path: str, mask_method: str, mask_params: dict):
    """Test mask creation on a single image."""
    image = cv2.imread(image_path)
    if image is None:
        print(f"Could not read image: {image_path}")
        return
    
    mask_func = MASK_FUNCTIONS[mask_method]
    mask = mask_func(image, mask_params)
    
    # Display results
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    original_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    axes[0].imshow(original_rgb)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Mask')
    axes[1].axis('off')
    
    # Overlay mask on original
    overlay = original_rgb.copy()
    mask_colored = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
    mask_colored_rgb = cv2.cvtColor(mask_colored, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(overlay, 0.7, mask_colored_rgb, 0.3, 0)
    axes[2].imshow(overlay)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return mask

# Uncomment and modify to test on a single image:
# test_image_path = "path/to/test/image.jpg"
# test_single_image(test_image_path, MASK_METHOD, MASK_PARAMS)


## 6. Tips and Examples

### Example Configurations:

**For threshold-based masking:**
```python
MASK_METHOD = 'threshold'
MASK_PARAMS = {
    'threshold_value': 127,
    'threshold_type': cv2.THRESH_BINARY,
}
```

**For color-based masking (e.g., detect green objects):**
```python
MASK_METHOD = 'color_range'
MASK_PARAMS = {
    'lower_hsv': np.array([40, 50, 50]),  # Lower green
    'upper_hsv': np.array([80, 255, 255]),  # Upper green
}
```

**For custom naming (e.g., add timestamp or category):**
```python
def mask_naming_function(original_filename: str) -> str:
    name, ext = os.path.splitext(original_filename)
    category = name.split('_')[0]  # Extract category from filename
    return f"{category}_mask_{name}{ext}"
```
